# Synthetic Data Generator

In this notebook, we will show how to randomly generate synthetic labeled dataset for training Neural Networks.

For both models (Heston and rBergomi), the dataset should consist of ```features``` and ```labels```.

For the ```features```, we randomly draw from some specific distributions proposed by the paper; for the ```labels```, we use pricer functions to compute the IV.

**NB**: The pricer of Heston model is implemented by the package ```QuantLib```, we use it to save our life, while the pricer of rBergomi model, we refer to the implementation of (https://github.com/amuguruza/RoughFCLT/blob/master/rDonsker.ipynb) and (https://github.com/ryanmccrickerd/rough_bergomi). The pricer of rBergomi is based on a Monte-Carlo simulation, so it is extreeeeeeemly slow. So instead of generating $10^6$ samples as proposed in the paper, we only managed to generate $\sim 5 \times 10^5$.

## Generate Heston data

In [1]:
import os
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from tqdm import tqdm
tqdm.pandas()

import QuantLib as ql

In [2]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
from py_lets_be_rational.exceptions import BelowIntrinsicException
import sklearn.utils
import logging

logging.basicConfig(format="%(asctime)s - %(levelname)s - %(message)s", level=logging.INFO)

### Define the Hestong pricer by using ```QuantLib```

In [3]:
def heston_pricer(lambd, vbar, eta, rho, v0, r, q, tau, S0, K):
    """Computes European Call price under Heston dynamics with closedform solution.
    
    Parameters:
    -----------
        lambd: mean-reversion speed
        vbar: long-term average variance
        eta: volatility of variance
        rho: correlation between stock and vol
        v0: spot variance
        r: risk-free interest rate
        q: dividend rate
        tau: time to maturity in years (365 trading days per year)
        S0: initial spot price
        K: strike price
    """
    today = datetime.date.today()
    ql_date = ql.Date(today.day, today.month, today.year)
    day_count = ql.Actual365Fixed()
    ql.Settings.instance().evaluationDate = ql_date
    
    # option data
    option_type = ql.Option.Call
    payoff = ql.PlainVanillaPayoff(option_type, K)
    maturity_date = ql_date + int(round(tau * 365))
    exercise = ql.EuropeanExercise(maturity_date)
    european_option = ql.VanillaOption(payoff, exercise)
    
    # Heston process
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S0))
    flat_ts = ql.YieldTermStructureHandle(ql.FlatForward(ql_date, r, day_count))
    dividend_yield = ql.YieldTermStructureHandle(ql.FlatForward(ql_date, q, day_count))
    heston_process = ql.HestonProcess(flat_ts, dividend_yield, spot_handle, v0, lambd, vbar, eta, rho)
    
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    
    # check numerical stability
    try:
        price = european_option.NPV()
        if price <= 0 or price + K < S0:
            iv = np.nan
            logging.debug("NumStabProblem: Price {}. Intrinsic {}. Time {}. Strike {}.".format(price, S0-K, tau, K))
        else:
            logging.debug("Success: Price {} > intrinsic {}".format(price, S0-K))
            iv = implied_volatility(price, S0, K, tau, r, 'c')
    except RuntimeError:
        logging.info("RuntimeError: Intrinsic {}. Time {}. Strike {}.".format(S0-K, tau, K))
        price = np.nan
        iv = np.nan
    return price, iv

### Generate labeled data

Load previously generated $(m, T)$ pairs

In [4]:
K_T = pd.read_csv("./data/strike_maturity.csv", index_col=0)

In [5]:
K_T.shape

(1000000, 2)

In [6]:
K_T.head()

,Moneyness,Time to Maturity (years)
0,0.770725,0.241266
1,0.836500,0.236654
2,0.763395,0.248873
3,0.963026,0.201243
4,0.784699,0.227981


Initialize the data frame to store labeled data

In [7]:
columns = ['lambda', 'vbar', 'eta', 'rho', 'v0', 'iv']
data_nn = pd.DataFrame(index=K_T.index, columns=columns)
data_nn = pd.concat([K_T, data_nn], axis=1)

In [8]:
data_nn.shape

(1000000, 8)

In [9]:
data_nn.head()

,Moneyness,Time to Maturity (years),lambda,vbar,eta,rho,v0,iv
0,0.770725,0.241266,NaN,NaN,NaN,NaN,NaN,NaN
1,0.836500,0.236654,NaN,NaN,NaN,NaN,NaN,NaN
2,0.763395,0.248873,NaN,NaN,NaN,NaN,NaN,NaN
3,0.963026,0.201243,NaN,NaN,NaN,NaN,NaN,NaN
4,0.784699,0.227981,NaN,NaN,NaN,NaN,NaN,NaN


Specify model parameters' distribution. For Heston model, we draw the model parameter $\mu = (\eta, \rho, \lambda, \bar{v}, v_0)$ from uniform distribution as proposed in the Table 1 of the paper.

In [10]:
# PARAMETERS
n_samples = K_T.shape[0]

# Heston parameter, bounds by Moodley (2005)
lambd_bounds = [0, 10]
vbar_bounds = [0, 1]
eta_bounds = [0, 5]
rho_bounds = [-1, 0]
v0_bounds = [0, 1]

# Market params
S0 = 1
r = 0
q = 0

In [11]:
data_nn['lambda'] = np.random.uniform(lambd_bounds[0], lambd_bounds[1], n_samples)
data_nn['vbar'] = np.random.uniform(vbar_bounds[0], vbar_bounds[1], n_samples)
data_nn['eta'] = np.random.uniform(eta_bounds[0], eta_bounds[1], n_samples)
data_nn['rho'] = np.random.uniform(rho_bounds[0], rho_bounds[1], n_samples)
data_nn['v0'] = np.random.uniform(v0_bounds[0], v0_bounds[1], n_samples)

In [12]:
data_nn.head()

,Moneyness,Time to Maturity (years),lambda,vbar,eta,rho,v0,iv
0,0.770725,0.241266,7.289124,0.548211,3.390413,-0.165641,0.874642,NaN
1,0.836500,0.236654,8.956104,0.337098,4.198422,-0.938781,0.240779,NaN
2,0.763395,0.248873,5.407095,0.456241,4.359875,-0.287999,0.792108,NaN
3,0.963026,0.201243,1.877246,0.526145,4.863331,-0.441461,0.947103,NaN
4,0.784699,0.227981,5.745099,0.866694,2.537011,-0.155042,0.490160,NaN


Call ```heston_pricer``` to compute the IV

In [13]:
from tqdm import tqdm
tqdm.pandas()  # 为pandas启用进度条

data_nn['iv'] = data_nn.progress_apply(lambda row: heston_pricer(row['lambda'], row['vbar'], row['eta'], 
                                                        row['rho'], row['v0'], r, q, 
                                                        row['Time to Maturity (years)'], S0, row['Moneyness'])[1], axis=1)

  1%|▏         | 12807/1000000 [00:37<59:24, 276.96it/s]  2026-01-14 20:46:21,144 - INFO - RuntimeError: Intrinsic 0.06526726093513402. Time 0.21305951859074. Strike 0.934732739064866.
2026-01-14 20:46:21,424 - INFO - RuntimeError: Intrinsic 0.16779870873135716. Time 0.1809976093510713. Strike 0.8322012912686428.
  3%|▎         | 31899/1000000 [01:31<34:21, 469.59it/s]  2026-01-14 20:47:15,614 - INFO - RuntimeError: Intrinsic 0.15985727998417876. Time 0.2054481773717948. Strike 0.8401427200158212.
2026-01-14 20:47:15,921 - INFO - RuntimeError: Intrinsic 0.18639303106637128. Time 0.1992319727091182. Strike 0.8136069689336287.
  3%|▎         | 33481/1000000 [01:37<34:47, 462.95it/s]  2026-01-14 20:47:21,572 - INFO - RuntimeError: Intrinsic 0.21693736899028593. Time 0.1489368373523532. Strike 0.7830626310097141.
2026-01-14 20:47:21,832 - INFO - RuntimeError: Intrinsic 0.10754825557070813. Time 0.246113914272666. Strike 0.8924517444292919.
  4%|▎         | 37374/1000000 [01:49<36:27, 439.9

Drop ```NaN``` data

In [14]:
data_nn.dropna(inplace=True)

In [15]:
data_nn.shape

(998348, 8)

Split generated labeled data into ```train```, ```val``` and ```test```

In [16]:
# data_nn = data_nn.iloc[:990000, :]
data_nn.reset_index(drop=True, inplace=True)

data_train, data_val, data_test = np.split(data_nn, [int(9e5), int(9.45e5)], axis=0)
data_train.shape, data_val.shape, data_test.shape

((900000, 8), (45000, 8), (53348, 8))

Store splitted data to local files

In [ ]:
# data_train.to_csv("./data/heston/train.csv", index=False)

# train data
length1 = len(data_train) // 2
df1 = data_train.iloc[:length1, :]
df2 = data_train.iloc[length1:, :]
df1.to_csv('data/heston/train1.csv', index=False)
df2.to_csv('data/heston/train2.csv', index=False)

# val data
data_val.to_csv("./data/heston/val.csv", index=False)

# test data
data_test.to_csv("./data/heston/test.csv", index=False)

----------

----------

## Generate rBergomi data

In [1]:
import numpy as np
from matplotlib import pyplot as plt
from rbergomi.rbergomi import rBergomi

%matplotlib inline

Define rBergomi pricer with Cholesky decomposition method, and Monte Carlo simulation

In [2]:
def rBergomi_pricer(H, eta, rho, v0, tau, K, S0, MC_samples=40000):
    """Computes European Call price under rBergomi dynamics with MC sampling.
    
    Parameters:
    -----------
        H: Hurst parameter
        eta: volatility of variance
        rho: correlation between stock and vol
        v0: spot variance
        tau: time to maturity in years (365 trading days per year)
        K: strike price
    """
    try:
        rB = rBergomi(n=365, N=MC_samples, T=tau, a=H-0.5)
        dW1, dW2 = rB.dW1(), rB.dW2()
        Y = rB.Y(dW1)
        dB = rB.dB(dW1, dW2, rho)
        xi = v0
        V = rB.V(Y, xi, eta)
        S = rB.S(V, dB)
        ST = S[:, -1]
        price = np.mean(np.maximum(ST-K, 0))
    except:
        return np.nan, np.nan
    
    # check numerical stability
    if price <= 0 or price + K < S0:
        iv = np.nan
        logging.debug("NumStabProblem: Price {}. Intrinsic {}. Time {}. Strike {}.".format(price, S0-K, tau, K))
    else:
        logging.debug("Success: Price {} > intrinsic {}".format(price, S0-K))
        iv = implied_volatility(price, S0, K, tau, 0, 'c')
    return price, iv

### Generate rBergomi labeled data

Load previously generated $(m,T)$ data

In [200]:
K_T = pd.read_csv("./data/strike_maturity.csv", index_col=0)

In [201]:
K_T.shape

(1000000, 2)

In [202]:
K_T.head()

,Moneyness,Time to Maturity (years)
0,0.820165,0.219184
1,0.926490,0.212487
2,0.834318,0.190719
3,1.053977,0.226441
4,0.802749,0.208923


In [204]:
# PARAMETERS
n_samples = K_T.shape[0]

# Market params
S0 = 1.

Define rBergomi parameter generator with ```scipy.stats.truncnorm```

In [ ]:
from scipy.stats import truncnorm

def param_generator(H_generator=truncnorm(-1.2, 8.6, 0.07, 0.05), 
                    eta_generator=truncnorm(-3, 3, 2.5, 0.5), 
                    rho_generator=truncnorm(-0.25, 2.25, -0.95, 0.2), 
                    v0_generator=truncnorm(-2.5, 7, 0.3, 0.1)):
    rslt = {
        'H': H_generator.rvs(),
        'eta': eta_generator.rvs(),
        'rho': rho_generator.rvs(),
        'v0': v0_generator.rvs() ** 2
    }
    return rslt

In [205]:
def generate_rBergomi_sample(K, T, param_generator, S0=1.0):
    """ Generates a rBergomi sample with random parameters
    """
    counter = 0
    while counter < 10:
        params = param_generator()
        H, eta, rho, v0 = params['H'], params['eta'], params['rho'], params['v0']
        _, iv = rBergomi_pricer(H, eta, rho, v0, T, K, S0)
        if np.isnan(iv):
            counter += 1
        else:
            break
    else:
        logging.warning("Tried 10 times, none valid sample obtained.")
    sample = {
        'H': H,
        'eta': eta,
        'rho': rho,
        'v0': v0,
        'iv': iv
    }
    return sample

Generate labeled data

**NB**: The next cells are executable, but it takes too long a time to generate $10^6$ samples, so in practice we generated only half of that on a cloud-based virtual machine.

In [ ]:
data_nn = K_T.merge(K_T.progress_apply(
    lambda row: pd.Series(generate_rBergomi_sample(row['Moneyness'], row['Time to Maturity (years)'], param_generator, S0)), 
    axis=1), left_index=True, right_index=True)

In [ ]:
data_nn.dropna(inplace=True)

In [ ]:
data_nn.shape

In [ ]:
data_nn.head()

In [ ]:
data_nn.to_csv("./data/rBergomi/labeled_data_all.csv", index=False)

In [ ]:
# data_nn = data_nn.iloc[:990000, :]
data_nn.reset_index(drop=True, inplace=True)

# Dissecting labeled pairs into training, validation and testing sets.

data_train, data_val, data_test = np.split(data_nn, [int(9e5), int(9.5e5)], axis=0)

data_train.shape, data_val.shape, data_test.shape

In [ ]:
data_train.to_csv("./data/heston/train.csv", index=False)
data_val.to_csv("./data/heston/val.csv", index=False)
data_test.to_csv("./data/heston/test.csv", index=False)

# References

[1] https://github.com/ryanmccrickerd/rough_bergomi

[2] https://github.com/amuguruza/RoughFCLT/blob/master/rDonsker.ipynb